### Imports

In [11]:
from databricks.connect import DatabricksSession
import os

# Ensure these are actually set in your environment
spark = DatabricksSession.builder \
    .host(os.getenv("DATABRICKS_HOST")) \
    .token(os.getenv("DATABRICKS_TOKEN")) \
    .serverless() \
    .getOrCreate()

# Test it
spark.sql("SELECT 2").show()

+---+
|  2|
+---+
|  2|
+---+



In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI
from langchain.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent
from databricks.vector_search.client import VectorSearchClient

from pyspark.sql.functions import col, expr, pandas_udf

load_dotenv()


d:\lums-python-programming\databricks-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

# Bonus Task 1

In [8]:
def retrieve_formatted_context(question: str, topk: int = 3):
    results = VectorSearchClient().get_index(
                endpoint_name="24280021_vector_endpoint",
                index_name="24280021-pa3.vector_index.fixed_vector_index"
            ).similarity_search(
                    query_text=question, 
                    columns=["ChunkContent"],
                    num_results=topk
                )
    docs = results.get('result', {}).get('data_array', [])
    raw_docs = "\n\n".join([doc[0] for doc in docs if doc])
    return raw_docs


@tool
def vector_search_tool(query: str) -> str:
    """ 
    This function takes a user query as input, performs a similarity search against the specified vector store endpoint, and returns the most relevant chunks of text as context for answering the query.
    Args:  query (str): The user's question or query that needs to be answered using the vector store.
    Returns: str: A formatted string containing the most relevant chunks of text retrieved from the vector store based on the input query.
    """
    
    try:
        formatted_context = retrieve_formatted_context(question=query, topk=3)
        return formatted_context
    
    except Exception as e:
        return f"Database search failed: {e}"



@tool
def read_fallback_document(category: str) -> str:
    """ 
    This function takes a category as input, maps it to a corresponding file path, and attempts to read the content of the file. If the category is not found or if there is an error reading the file, it returns an appropriate error message.
    Args: category (str): The category for which the fallback document needs to be read.
    Returns: str: The content of the fallback document if found and read successfully, otherwise an error message indicating the issue.
    """

    file_map = {
        "databricks": "/Volumes/24280021-pa3/default/text_documents/databricks_info.txt",
        "azure": "/Volumes/24280021-pa3/default/text_documents/azure_info.txt",
    }
    
    file_path = file_map.get(category.lower())
    if not file_path:
        return f"No documentation found for category: {category}"
        
    try:        
        file_df = spark.read.format("binaryFile") \
            .load(file_path) \
            .select(col("content").cast("string"))
        full_text = file_df.first()["content"]
        return full_text
    except Exception as e:
        return f"Error reading document: {e}"
    

    

def build_stateful_mcp_agent():
    llm = AzureChatOpenAI(
        azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
        azure_deployment=os.environ.get("AZURE_OPENAI_CHAT_DEPLOYMENT"), 
        api_version=os.environ.get("AZURE_OPENAI_API_VERSION"), 
        temperature=0.0
    )
    
    tools = [vector_search_tool, read_fallback_document]
    memory = MemorySaver()
    
    system_prompt = (
        """ 
        You are a helpful, stateful technical assistant. 
        Always try to answer questions using the `vector_search_tool` first. 
        If the search results do not contain the answer, you must use the `read_fallback_document` tool.
        """
    )

    mcp_agent = create_react_agent(model=llm, 
                                   tools=tools, 
                                   checkpointer=memory, 
                                   prompt=system_prompt)
    return mcp_agent

agent = build_stateful_mcp_agent()

C:\Users\HP\AppData\Local\Temp\ipykernel_13272\3538278358.py:80: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  mcp_agent = create_react_agent(model=llm,


In [9]:
# Summary: Executes queries against the agent while maintaining a consistent session thread.
def execute_mcp_queries(mcp_agent, queries: list, thread_id: str):
    config = {"configurable": {"thread_id": "1"}}
    
    for query in queries:
        print(f"\nUser: {query}")
        payload = {"messages": [("user", query)]}
        
        response = mcp_agent.invoke(payload, config=config)
        final_message = response["messages"][-1].content
        
        print(f"Agent: {final_message}")

In [10]:
validation_queries = [
    "What tools do you have?",
    "What is searchAgent-X?",
    "What does this agent do specifically?",
    "Use your tool to search for information about Azure.",
    "Based on those search results, summarize its main features.",
    "What is Databricks?",
    "What core features differentiate both products?"
]

print("--- TESTING STATEFUL MCP AGENT ---")
execute_mcp_queries(agent, validation_queries, "mcp_test_session_01")

--- TESTING STATEFUL MCP AGENT ---

User: What tools do you have?
Agent: I have access to the following tools:

1. vector_search_tool: This tool allows me to perform a similarity search against a specified vector store endpoint to find the most relevant chunks of text related to a user's query.

2. read_fallback_document: This tool lets me read the content of a fallback document based on a given category if the vector search does not provide sufficient information.

If you have a question or need information, I will first try to use the vector_search_tool to find the best answer. If that doesn't work, I will use the read_fallback_document tool. How can I assist you today?

User: What is searchAgent-X?
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development onl